# 04_policy_gradient_manual.ipynb
Manual implementation tutorial: A2C and PPO for MiniAtari Breakout (PyTorch)
- Single-file, step-by-step
- Uses MinAtar environment
- Saves models, GIFs, and plots to Google Drive


In [ ]:
!pip install --quiet minatar imageio-ffmpeg

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = "/content/drive/MyDrive/rl-final-project"
os.makedirs(DRIVE, exist_ok=True)
os.makedirs(os.path.join(DRIVE, "policy_gradient"), exist_ok=True)
os.makedirs(os.path.join(DRIVE, "visuals", "ppo_manual"), exist_ok=True)
os.makedirs(os.path.join(DRIVE, "visuals", "a2c_manual"), exist_ok=True)
os.makedirs(os.path.join(DRIVE, "plots", "comparison"), exist_ok=True)
print("Drive folder:", DRIVE)

Mounted at /content/drive
Drive folder: /content/drive/MyDrive/rl-final-project


In [ ]:
import random, time, math, pickle
from collections import deque, namedtuple
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import imageio
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [ ]:
import minatar

class MiniAtariWrapper:
    """Simple wrapper: returns flattened normalized 10x10 (sum-channels -> 100) state"""
    def __init__(self, game="breakout"):
        self.env = minatar.Environment(game)
        self.n_actions = self.env.num_actions()
        self.state_shape = (100,)
    def reset(self):
        self.env.reset()
        st = self.env.state()
        return self._preprocess(st)
    def step(self, action):
        # MinAtar .act returns (reward, done)
        r, done = self.env.act(int(action))
        st = self.env.state()
        return self._preprocess(st), float(r), bool(done)
    def _preprocess(self, st):
        arr = np.array(st, dtype=np.float32)
        # sum channels -> 2D
        if arr.ndim == 3:
            arr = arr.sum(axis=2)
        mn, mx = arr.min(), arr.max()
        rng = mx - mn if mx > mn else 1.0
        arr = (arr - mn) / rng
        flat = arr.flatten()
        if flat.shape[0] != 100:
            flat = np.resize(flat, 100)
        return flat
    def render_frame(self):
        st = self.env.state()
        arr = np.array(st, dtype=np.float32)
        if arr.ndim == 3:
            img = arr.sum(axis=2)
        else:
            img = arr
        mn, mx = img.min(), img.max()
        rng = mx - mn if mx > mn else 1.0
        img = ((img - mn) / rng * 255).astype(np.uint8)
        return img

In [ ]:
def save_pickle(obj, path):
    with open(path, "wb") as f:
        pickle.dump(obj, f)

def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def make_gif(frames, path, fps=12):
    imageio.mimsave(path, frames, fps=fps)
    print("Saved GIF ->", path)

def eval_policy(env_wrapper, policy_fn, episodes=10, steps=400):
    scores = []
    for ep in range(episodes):
        s = env_wrapper.reset()
        total = 0.0
        for t in range(steps):
            a = policy_fn(s)
            s, r, done = env_wrapper.step(a)
            total += r
            if done:
                break
        scores.append(total)
    return np.array(scores)

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_sizes=[256,128], output_dim=None):
        super().__init__()
        layers = []
        last = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(last, h))
            layers.append(nn.ReLU())
            last = h
        if output_dim is not None:
            layers.append(nn.Linear(last, output_dim))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

In [ ]:
class Actor(nn.Module):
    def __init__(self, input_dim, hidden=[256,128], n_actions=6):
        super().__init__()
        self.mlp = MLP(input_dim, hidden, output_dim=n_actions)
    def forward(self, x):
        return torch.softmax(self.mlp(x), dim=-1)  # probabilities

class Critic(nn.Module):
    def __init__(self, input_dim, hidden=[256,128]):
        super().__init__()
        self.mlp = MLP(input_dim, hidden, output_dim=1)
    def forward(self, x):
        return self.mlp(x).squeeze(-1)  # state-value

In [ ]:
def compute_returns_and_advantages(rewards, values, dones, gamma=0.99, lam=0.95):
    # Generalized Advantage Estimation (GAE)
    T = len(rewards)
    advantages = np.zeros(T, dtype=np.float32)
    lastgaelam = 0
    # values: length T+1 (bootstrap)
    for t in reversed(range(T)):
        delta = rewards[t] + gamma * values[t+1] * (1 - dones[t]) - values[t]
        advantages[t] = lastgaelam = delta + gamma * lam * (1 - dones[t]) * lastgaelam
    returns = advantages + values[:-1]
    return returns, advantages

def train_a2c(env_wrapper,
              total_steps=100000,
              n_steps=5,          # steps per update (A2C multi-step)
              gamma=0.99,
              lam=0.95,
              lr=7e-4,
              entropy_coef=0.01,
              value_coef=0.5,
              max_grad_norm=0.5,
              hidden=[256,128]):

    obs_dim = env_wrapper.state_shape[0] if hasattr(env_wrapper, "state_shape") else 100
    n_actions = env_wrapper.n_actions

    actor = Actor(obs_dim, hidden, n_actions).to(device)
    critic = Critic(obs_dim, hidden).to(device)

    opt = optim.Adam(list(actor.parameters()) + list(critic.parameters()), lr=lr)

    obs = env_wrapper.reset()
    num_steps = 0
    episode_rewards = []
    log_interval = max(1, total_steps // 50)
    losses = []

    while num_steps < total_steps:
        # collect rollout
        states = []
        actions = []
        rewards = []
        dones = []
        values = []

        for step in range(n_steps):
            s_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
            with torch.no_grad():
                pi = actor(s_t)
                v = critic(s_t)
            pi = pi.cpu().numpy().squeeze(0)
            a = np.random.choice(len(pi), p=pi)

            states.append(obs.copy())
            actions.append(a)
            values.append(v.item())

            obs, r, done = env_wrapper.step(a)
            rewards.append(r)
            dones.append(float(done))
            num_steps += 1

            if done:
                obs = env_wrapper.reset()

        # bootstrap value
        s_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            v_last = critic(s_t).item()
        values.append(v_last)

        # compute returns/advantages (numpy)
        returns, advantages = compute_returns_and_advantages(rewards, values, dones, gamma=gamma, lam=lam)

        # convert to tensors
        states_t = torch.tensor(np.stack(states), dtype=torch.float32).to(device)
        actions_t = torch.tensor(actions, dtype=torch.long).to(device)
        returns_t = torch.tensor(returns, dtype=torch.float32).to(device)
        advantages_t = torch.tensor(advantages, dtype=torch.float32).to(device)

        # forward
        logits = actor(states_t)  # probs (B, A)
        dist = torch.distributions.Categorical(logits)
        log_probs = dist.log_prob(actions_t)
        entropy = dist.entropy().mean()
        values_pred = critic(states_t)

        # losses
        policy_loss = -(log_probs * advantages_t).mean()
        value_loss = value_coef * (returns_t - values_pred).pow(2).mean()
        ent_loss = -entropy_coef * entropy

        loss = policy_loss + value_loss + ent_loss

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(list(actor.parameters()) + list(critic.parameters()), max_grad_norm)
        opt.step()

        losses.append(loss.item())

        if num_steps % log_interval == 0:
            print(f"[A2C] Steps {num_steps}/{total_steps} Loss {np.mean(losses[-50:]):.4f}")

    # return trained models
    return actor, critic

In [ ]:
class PPOBuffer:
    def __init__(self):
        self.states = []
        self.actions = []
        self.rewards = []
        self.dones = []
        self.log_probs = []
        self.values = []
    def clear(self):
        self.__init__()

def train_ppo(env_wrapper,
              total_steps=200000,
              batch_size=64,
              update_steps=2048,
              epochs=10,
              gamma=0.99,
              lam=0.95,
              clip_eps=0.2,
              lr=3e-4,
              ent_coef=0.0,
              vf_coef=0.5,
              max_grad_norm=0.5,
              hidden=[256,128]):

    obs_dim = env_wrapper.state_shape[0] if hasattr(env_wrapper, "state_shape") else 100
    n_actions = env_wrapper.n_actions

    # actor as logits (not directly softmax in network)
    policy_net = MLP(obs_dim, hidden, output_dim=n_actions).to(device)
    value_net = MLP(obs_dim, hidden, output_dim=1).to(device)

    opt = optim.Adam(list(policy_net.parameters()) + list(value_net.parameters()), lr=lr)

    buffer = PPOBuffer()
    obs = env_wrapper.reset()
    steps_done = 0
    log_every = max(1, total_steps // 50)
    losses = []

    while steps_done < total_steps:
        # collect update_steps transitions
        for _ in range(update_steps):
            s_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
            with torch.no_grad():
                logits = policy_net(s_t)
                probs = torch.softmax(logits, dim=-1)
                dist = torch.distributions.Categorical(probs)
                a = dist.sample().item()
                lp = dist.log_prob(torch.tensor(a)).item()
                v = value_net(s_t).item()

            buffer.states.append(obs.copy())
            buffer.actions.append(a)
            buffer.log_probs.append(lp)
            buffer.values.append(v)

            obs, r, done = env_wrapper.step(a)
            buffer.rewards.append(r)
            buffer.dones.append(float(done))
            steps_done += 1

            if done:
                obs = env_wrapper.reset()

        # compute returns & advantages using GAE
        values = np.array(buffer.values + [value_net(torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)).item()])
        rewards = np.array(buffer.rewards)
        dones = np.array(buffer.dones)
        returns, advantages = compute_returns_and_advantages(rewards, values, dones, gamma=gamma, lam=lam)

        # convert to tensors
        states_t = torch.tensor(np.stack(buffer.states), dtype=torch.float32).to(device)
        actions_t = torch.tensor(buffer.actions, dtype=torch.long).to(device)
        old_log_probs_t = torch.tensor(buffer.log_probs, dtype=torch.float32).to(device)
        returns_t = torch.tensor(returns, dtype=torch.float32).to(device)
        advs_t = torch.tensor(advantages, dtype=torch.float32).to(device)
        advs_t = (advs_t - advs_t.mean()) / (advs_t.std() + 1e-8)

        # optimization epochs (minibatches)
        n_samples = states_t.shape[0]
        inds = np.arange(n_samples)
        for _ in range(epochs):
            np.random.shuffle(inds)
            for start in range(0, n_samples, batch_size):
                mb_inds = inds[start:start+batch_size]
                mb_states = states_t[mb_inds]
                mb_actions = actions_t[mb_inds]
                mb_oldlp = old_log_probs_t[mb_inds]
                mb_returns = returns_t[mb_inds]
                mb_advs = advs_t[mb_inds]

                logits = policy_net(mb_states)
                probs = torch.softmax(logits, dim=-1)
                dist = torch.distributions.Categorical(probs)
                mb_logp = dist.log_prob(mb_actions)
                mb_entropy = dist.entropy().mean()

                mb_values = value_net(mb_states).squeeze(-1)

                ratio = torch.exp(mb_logp - mb_oldlp)
                surr1 = ratio * mb_advs
                surr2 = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * mb_advs
                policy_loss = -torch.min(surr1, surr2).mean()
                value_loss = vf_coef * (mb_returns - mb_values).pow(2).mean()
                entropy_loss = -ent_coef * mb_entropy

                loss = policy_loss + value_loss + entropy_loss

                opt.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(list(policy_net.parameters()) + list(value_net.parameters()), max_grad_norm)
                opt.step()
                losses.append(loss.item())

        buffer.clear()

        if steps_done % log_every == 0:
            print(f"[PPO] Steps {steps_done}/{total_steps} Loss {np.mean(losses[-200:]):.4f}")

    return policy_net, value_net

In [ ]:
def save_model(model, path):
    torch.save(model.state_dict(), path)
    print("Saved:", path)

def load_model(model_class, path, *args, **kwargs):
    model = model_class(*args, **kwargs).to(device)
    model.load_state_dict(torch.load(path, map_location=device))
    model.eval()
    return model

In [ ]:
# Create wrapper
env_pg = MiniAtariWrapper()

# === A2C quick run (test) ===
actor, critic = train_a2c(env_pg, total_steps=30000, n_steps=5, lr=7e-4)
save_model(actor, os.path.join(DRIVE, "policy_gradient", "a2c_actor.pth"))
save_model(critic, os.path.join(DRIVE, "policy_gradient", "a2c_critic.pth"))

# === PPO quick run (test) ===
policy_net, value_net = train_ppo(env_pg, total_steps=50000, update_steps=1024, epochs=6, batch_size=64, lr=3e-4)
save_model(policy_net, os.path.join(DRIVE, "policy_gradient", "ppo_policy.pth"))
save_model(value_net, os.path.join(DRIVE, "policy_gradient", "ppo_value.pth"))


[A2C] Steps 600/30000 Loss 0.0998
[A2C] Steps 1200/30000 Loss 0.0902
[A2C] Steps 1800/30000 Loss 0.1053
[A2C] Steps 2400/30000 Loss 0.0227
[A2C] Steps 3000/30000 Loss 0.0391
[A2C] Steps 3600/30000 Loss -0.0007
[A2C] Steps 4200/30000 Loss 0.0045
[A2C] Steps 4800/30000 Loss 0.0547
[A2C] Steps 5400/30000 Loss 0.0753
[A2C] Steps 6000/30000 Loss 0.0884
[A2C] Steps 6600/30000 Loss 0.0444
[A2C] Steps 7200/30000 Loss -0.0007
[A2C] Steps 7800/30000 Loss 0.0563
[A2C] Steps 8400/30000 Loss -0.0458
[A2C] Steps 9000/30000 Loss 0.0237
[A2C] Steps 9600/30000 Loss 0.1273
[A2C] Steps 10200/30000 Loss 0.0811
[A2C] Steps 10800/30000 Loss 0.0356
[A2C] Steps 11400/30000 Loss -0.0010
[A2C] Steps 12000/30000 Loss -0.0364
[A2C] Steps 12600/30000 Loss 0.0017
[A2C] Steps 13200/30000 Loss -0.0297
[A2C] Steps 13800/30000 Loss -0.1024
[A2C] Steps 14400/30000 Loss -0.0378
[A2C] Steps 15000/30000 Loss -0.0292
[A2C] Steps 15600/30000 Loss 0.0682
[A2C] Steps 16200/30000 Loss -0.0744
[A2C] Steps 16800/30000 Loss -0.024

In [ ]:
def rollout_actor_policy(actor_model, env_wrapper, steps=400, scale=20):
    frames = []
    s = env_wrapper.reset()
    for _ in range(steps):
        with torch.no_grad():
            probs = actor_model(torch.tensor(s, dtype=torch.float32).unsqueeze(0).to(device))
            probs = probs.cpu().numpy().squeeze(0)
        a = np.argmax(probs)  # greedy for visualization
        s, r, done = env_wrapper.step(a)
        img = env_wrapper._preprocess(env_wrapper.env.state())  # normalized flat
        img2 = (img.reshape(10,10) * 255).astype(np.uint8)
        frame = Image.fromarray(img2).resize((10*scale,10*scale), Image.NEAREST)
        frames.append(np.array(frame))
        if done:
            s = env_wrapper.reset()
    return frames

def rollout_policy_net(policy_net, env_wrapper, steps=400, scale=20):
    frames = []
    s = env_wrapper.reset()
    for _ in range(steps):
        with torch.no_grad():
            logits = policy_net(torch.tensor(s, dtype=torch.float32).unsqueeze(0).to(device))
            probs = torch.softmax(logits, dim=-1).cpu().numpy().squeeze(0)
        a = np.argmax(probs)
        s, r, done = env_wrapper.step(a)
        img = env_wrapper._preprocess(env_wrapper.env.state())
        img2 = (img.reshape(10,10) * 255).astype(np.uint8)
        frame = Image.fromarray(img2).resize((10*scale,10*scale), Image.NEAREST)
        frames.append(np.array(frame))
        if done:
            s = env_wrapper.reset()
    return frames

In [ ]:
def compare_and_save(env_wrapper, actor_model=None, policy_net=None, expert_fn=None, label_actor="A2C", label_policy="PPO"):
    # Evaluate
    scores = {}
    if actor_model is not None:
        def act_fn_a2c(s):
            with torch.no_grad():
                probs = actor_model(torch.tensor(s, dtype=torch.float32).unsqueeze(0).to(device)).cpu().numpy().squeeze(0)
            return int(np.argmax(probs))
        a2c_scores = eval_policy(env_wrapper, act_fn_a2c, episodes=20)
        scores[label_actor] = a2c_scores
    if policy_net is not None:
        def act_fn_ppo(s):
            with torch.no_grad():
                logits = policy_net(torch.tensor(s, dtype=torch.float32).unsqueeze(0).to(device))
                probs = torch.softmax(logits, dim=-1).cpu().numpy().squeeze(0)
            return int(np.argmax(probs))
        ppo_scores = eval_policy(env_wrapper, act_fn_ppo, episodes=20)
        scores[label_policy] = ppo_scores
    if expert_fn is not None:
        exp_scores = eval_policy(env_wrapper, expert_fn, episodes=20)
        scores["Expert"] = exp_scores

    # Boxplot
    labels = list(scores.keys())
    data = [scores[k] for k in labels]
    plt.figure(figsize=(8,5))
    plt.boxplot(data, labels=labels)
    plt.title("Policy comparison")
    out = os.path.join(DRIVE, "plots", "comparison", "ppo_a2c_comparison_box.png")
    plt.savefig(out, dpi=150)
    plt.close()
    print("Saved boxplot:", out)

    # Bar mean/std
    means = [np.mean(d) for d in data]
    stds = [np.std(d) for d in data]
    plt.figure(figsize=(6,4))
    plt.bar(labels, means, yerr=stds, capsize=8)
    plt.title("Mean reward ± std")
    out2 = os.path.join(DRIVE, "plots", "comparison", "ppo_a2c_comparison_bar.png")
    plt.savefig(out2, dpi=150)
    plt.close()
    print("Saved bar chart:", out2)

    return scores

In [ ]:
env_vis = MiniAtariWrapper()

# Load A2C actor
a2c_actor = Actor(100, [256,128], env_vis.n_actions).to(device)
a2c_actor.load_state_dict(torch.load(os.path.join(DRIVE, "policy_gradient", "a2c_actor.pth"), map_location=device))
a2c_actor.eval()

frames = rollout_actor_policy(a2c_actor, env_vis)
outpath = os.path.join(DRIVE, "visuals", "a2c_manual", "a2c_manual.gif")
make_gif(frames, outpath)

Saved GIF -> /content/drive/MyDrive/rl-final-project/visuals/a2c_manual/a2c_manual.gif


In [ ]:
env_vis = MiniAtariWrapper()

# Load PPO policy
ppo_policy = MLP(100, [256,128], 6).to(device)
ppo_policy.load_state_dict(torch.load(os.path.join(DRIVE, "policy_gradient", "ppo_policy.pth"), map_location=device))
ppo_policy.eval()

frames = rollout_policy_net(ppo_policy, env_vis)
outpath = os.path.join(DRIVE, "visuals", "ppo_manual", "ppo_manual.gif")
make_gif(frames, outpath)

Saved GIF -> /content/drive/MyDrive/rl-final-project/visuals/ppo_manual/ppo_manual.gif


In [ ]:
env_eval = MiniAtariWrapper()

# A2C eval
def a2c_policy(s):
    with torch.no_grad():
        p = a2c_actor(torch.tensor(s, dtype=torch.float32).unsqueeze(0).to(device))
    return int(torch.argmax(p).item())

a2c_scores = eval_policy(env_eval, a2c_policy, episodes=20)

# PPO eval
def ppo_policy_fn(s):
    with torch.no_grad():
        logits = ppo_policy(torch.tensor(s, dtype=torch.float32).unsqueeze(0).to(device))
        probs = torch.softmax(logits, dim=-1)
    return int(torch.argmax(probs).item())

ppo_scores = eval_policy(env_eval, ppo_policy_fn, episodes=20)

print("A2C:", np.mean(a2c_scores), np.std(a2c_scores))
print("PPO:", np.mean(ppo_scores), np.std(ppo_scores))

A2C: 3.15 0.9096702699330127
PPO: 5.4 1.2


In [ ]:
plt.figure(figsize=(7,5))
plt.boxplot([a2c_scores, ppo_scores], labels=["A2C", "PPO"])
plt.title("A2C vs PPO Performance Comparison (20 episodes)")
plt.ylabel("Episode Return")
plt.grid(True)

save_path = os.path.join(DRIVE, "plots", "comparison", "a2c_vs_ppo_box.png")
plt.savefig(save_path, dpi=150)
plt.close()

print("Saved:", save_path)

/tmp/ipython-input-3896718805.py:2: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot([a2c_scores, ppo_scores], labels=["A2C", "PPO"])


Saved: /content/drive/MyDrive/rl-final-project/plots/comparison/a2c_vs_ppo_box.png


In [ ]:
means = [np.mean(a2c_scores), np.mean(ppo_scores)]
stds = [np.std(a2c_scores), np.std(ppo_scores)]

plt.figure(figsize=(7,5))
plt.bar(["A2C", "PPO"], means, yerr=stds, capsize=8)
plt.title("A2C vs PPO Mean Reward ± Std")
plt.ylabel("Mean Episode Reward")

save_path = os.path.join(DRIVE, "plots", "comparison", "a2c_vs_ppo_bar.png")
plt.savefig(save_path, dpi=150)
plt.close()

print("Saved:", save_path)

Saved: /content/drive/MyDrive/rl-final-project/plots/comparison/a2c_vs_ppo_bar.png
